In [82]:
import html
import re
from collections import Counter
from copy import deepcopy

import nltk
import pandas as pd
from bertopic import BERTopic
from IPython.display import HTML, display
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import NMF, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer
from bertopic.dimensionality import BaseDimensionalityReduction
from umap import UMAP


In [52]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("averaged_perceptron_tagger_eng")
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

def preprocess_lyrics(text):
    if pd.isna(text):
        return ""

    text = text.lower()
    text = re.sub(r"\b\w*\d\w*\b", "", text)
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = word_tokenize(text)
    stems = [
        stemmer.stem(token)
        for token in tokens
        if token not in stop_words
    ]

    return " ".join(stems)

def get_word_count(text: str) -> dict:
    tokens = text.split()
    return Counter(token for token in tokens).most_common()

def get_word_count_df(text: str, top_n: int = 10) -> pd.DataFrame:
    tokens = text.split()
    most_common = Counter(tokens).most_common(top_n)

    return pd.DataFrame(
        most_common,
        columns=["token", "count"]
    )

def get_similarity_result(X_count, selected_indices) -> pd.DataFrame:
    results_count = []

    for idx in selected_indices:
        similarities = cosine_similarity(X_count[idx],X_count).flatten()

        similarities[idx] = -1

        most_similar_position = similarities.argmax()
        similarity_score = similarities[most_similar_position]

        results_count.append({
            "selected_index": idx,
            "similar_index": most_similar_position,
            "similarity": similarity_score
        })

    return pd.DataFrame(results_count).sort_values("similarity", ascending=False).reset_index(drop=True)

def display_lyrics_side_by_side(title_1, lyrics_1, title_2, lyrics_2):
    lyrics_1 = html.escape(str(lyrics_1))
    lyrics_2 = html.escape(str(lyrics_2))
    title_1 = html.escape(str(title_1))
    title_2 = html.escape(str(title_2))

    table = f"""
    <table style="width: 50%; table-layout: fixed;">
        <thead>
            <tr>
                <th style="width: 50%; text-align: left;">
                    {title_1}
                </th>
                <th style="width: 50%; text-align: left;">
                    {title_2}
                </th>
            </tr>
        </thead>
        <tbody>
            <tr>
                <td style="vertical-align: top; padding: 12px; text-align: left;">
                    <pre style="white-space: pre-wrap;
                                overflow-wrap: anywhere;
                                font-family: inherit;">{lyrics_1}</pre>
                </td>
                <td style="vertical-align: top; padding: 12px; text-align: left;">
                    <pre style="white-space: pre-wrap;
                                overflow-wrap: anywhere;
                                font-family: inherit;">{lyrics_2}</pre>
                </td>
            </tr>
        </tbody>
    </table>
    """

    display(HTML(table))

def display_word_counts_side_by_side(name_1, counts_1, name_2, counts_2):
    table_1 = counts_1.to_html(index=False)
    table_2 = counts_2.to_html(index=False)

    html_output = f"""
    <div style="display: flex; gap: 10px; align-items: flex-start;">
        <div>
            <h4>{name_1}</h4>
            {table_1}
        </div>
        <div>
            <h4>{name_2}</h4>
            {table_2}
        </div>
    </div>
    """

    display(HTML(html_output))

def lyric_pair_summary(df, row):
    selected_index = int(row["selected_index"])
    similar_index = int(row["similar_index"])

    original = df.iloc[selected_index]
    similar = df.iloc[similar_index]

    original_count = get_word_count_df(original["preprocessed_lyrics"])
    similar_count = get_word_count_df(similar["preprocessed_lyrics"])
    display_word_counts_side_by_side(
        original["name"], original_count, similar["name"], similar_count
    )

    display_lyrics_side_by_side(
        original["name"], original["lyrics"], similar["name"], similar["lyrics"]
    )
    

[nltk_data] Downloading package punkt to /home/arthurpmrs/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/arthurpmrs/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/arthurpmrs/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/arthurpmrs/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [3]:
# Obter letras pre-processadas
df = pd.read_csv("data/songs_reduced.csv")
df["preprocessed_lyrics"] = df["lyrics"].apply(preprocess_lyrics)
texts = df["preprocessed_lyrics"].fillna("")

In [ ]:
# Calcular matriz de documentos
# TODO: Justificar os parâmetros.
tfidf_vectorizer = TfidfVectorizer(
    norm="l2",
    max_df=0.9,
    min_df=10,
    stop_words="english"
)

X_tfidf = tfidf_vectorizer.fit_transform(texts)

print(X_tfidf.shape)
feature_names = tfidf_vectorizer.get_feature_names_out()
pd.DataFrame(X_tfidf.toarray(), columns=feature_names)

(20000, 8997)


,aa,aaah,aah,aaliyah,ab,abandon,abc,abel,abid,abil,...,필요,하고,하나,하는,하늘,하지,하지만,한번,함께,해도
0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.000000,0.0,0.0,0.108462,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19996,0.0,0.0,0.275023,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19997,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19998,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## a) NMF - Definir a quantidade de tópicos

Nessa etapa, foram testados diferentes quantidades de tópicos, com o objetivo de identificar qual seria a quantidade alvo de tópicos mais adequada para o dataset. A obtenção dos tópicos foi atingida por meio do modelo NMF, pois é uma boa opção para tratar o output do TF-IDF, sugerido pela questão. Os valores de número alvo de tópicos testados foram 3, 4, 5, 7 e 9, dos quais 7 apresentou uma boa diversificação de temas/estruturas textuais.

In [53]:
def get_nmf_topics(model, feature_names, top_n=8) -> pd.DataFrame:
    topics = []

    for topic_id, weights in enumerate(model.components_):
        top_indices = weights.argsort()[-top_n:][::-1]
        top_words = feature_names[top_indices]

        topics.append({
            "topic": topic_id,
            "top_words": ", ".join(top_words)
        })

    return pd.DataFrame(topics)

def get_nmf_topics_from_matrix(X, n_topics, feature_names, top_n=8):
    nmf_model = NMF(n_components=n_topics, random_state=42, max_iter=500)
    document_topic_weights = nmf_model.fit_transform(X)
    return document_topic_weights, get_nmf_topics(nmf_model, feature_names, top_n)

In [54]:
n_topics_candidates = [3, 4, 5, 7, 9]

for n_topics in n_topics_candidates:
    _, topics = get_nmf_topics_from_matrix(X_tfidf, n_topics, feature_names)
    display(topics)

,topic,top_words
0,0,"im, know, time, dont, ill, come, feel, ive"
1,1,"im, nigga, yeah, got, na, like, aint, fuck"
2,2,"love, oh, babi, heart, girl, yeah, na, know"


,topic,top_words
0,0,"im, know, time, dont, ill, come, feel, ive"
1,1,"nigga, yeah, im, got, bitch, like, fuck, aint"
2,2,"love, oh, babi, yeah, heart, girl, ooh, know"
3,3,"na, wan, gon, dont, babi, im, oh, know"


,topic,top_words
0,0,"im, know, time, dont, ill, come, feel, ive"
1,1,"nigga, im, got, yeah, bitch, fuck, like, aint"
2,2,"love, babi, heart, girl, true, know, let, need"
3,3,"na, wan, gon, dont, im, babi, know, girl"
4,4,"oh, yeah, babi, ooh, girl, whoa, want, know"


,topic,top_words
0,0,"time, come, day, away, life, night, ill, heart"
1,1,"nigga, got, bitch, yeah, fuck, shit, aint, like"
2,2,"love, babi, heart, true, girl, let, make, need"
3,3,"na, wan, gon, dont, babi, tonight, let, girl"
4,4,"oh, yeah, babi, ooh, girl, whoa, want, hey"
5,5,"mi, dem, di, yuh, fi, nuh, ah, gyal"
6,6,"im, dont, know, want, feel, think, got, say"


,topic,top_words
0,0,"time, come, day, away, life, night, ill, heart"
1,1,"nigga, bitch, got, fuck, shit, aint, like, em"
2,2,"love, babi, heart, true, girl, make, let, forev"
3,3,"na, wan, gon, dont, tonight, babi, danc, let"
4,4,"oh, babi, ooh, whoa, girl, lord, ohoh, littl"
5,5,"mi, dem, di, yuh, fi, nuh, ah, gyal"
6,6,"dont, know, want, got, need, say, tell, think"
7,7,"im, ive, feel, like, think, caus, ill, gon"
8,8,"yeah, babi, ooh, girl, got, uh, ayi, hey"


Com base nos resultados apresentados anteriormente, escolhemos uma quantidade de tópicos igual a 7 como alvo do modelo. O teste mostra que uma quantidade muito pequena, como 3, traz tópicos muito abrangentes. No caso do 7, podemos extrair significados diferentes a partir do conjunto de palavras de maior impacto.

- 0: Música sobre a passagem do tempo, amadurecimento
- 1: Vocabulário do Rap/Hip-Hop, conflito
- 2: Música sobre amor e relacionamento
- 3: Música sobre festejar, envolvendo música e dança
- 4: Música com muitos refrões, típico do Pop com bastante linguagem coloquial
- 5: Vocabulário Jamaicano
- 6: Música com tema introspectivo, sobre pensamentos

## b) NMF - Identificar as 5 palavras mais relevantes de cada tópico
A Tabela abaixo mostra as 5 palalvras de maior impacto para cada um dos 7 tópicos.

In [56]:
document_topic_weights, topics = get_nmf_topics_from_matrix(X_tfidf, 7, feature_names, top_n=5)
display(topics)

,topic,top_words
0,0,"time, come, day, away, life"
1,1,"nigga, got, bitch, yeah, fuck"
2,2,"love, babi, heart, true, girl"
3,3,"na, wan, gon, dont, babi"
4,4,"oh, yeah, babi, ooh, girl"
5,5,"mi, dem, di, yuh, fi"
6,6,"im, dont, know, want, feel"


## c) NMF - Identifique o tópico mais relevante de 5 documentos quaisquer

In [64]:
# Selecionar 5 músicas aleatoriamente
selected_indices = df.sample(5, random_state=21).index
selected = df.loc[selected_indices]
selected[["artists", "name", "preprocessed_lyrics"]]

,artists,name,preprocessed_lyrics
7834,"[""Phosphorescent""]",A New Anhedonia - Live,colder night came call day howl midnight call ...
8138,"[""Willie Nelson"", ""Snoop Dogg"", ""Kris Kristoff...",Roll Me Up,roll smoke die anyon dont like look em eye say...
7782,"[""Luke Combs""]",The Kind of Love We Make,weve burnin end keepin light ive thinkin need ...
2432,"[""Scott H. Biram""]",Lost Case Of Being Found,let tell bout one hors town like handful come ...
1708,"[""The Smashing Pumpkins""]",Galapogos - Remastered 2012,aint funni pretend still child softli stolen b...


In [65]:
# Selecionar o tópico mais adequado para cada uma das músicas
selected_documents = df.iloc[selected_indices].copy()
selected_topic_weights = document_topic_weights[selected_indices]
selected_documents["nmf_topic"] = (selected_topic_weights.argmax(axis=1))

selected_documents["topic_words"] = (
    selected_documents["nmf_topic"]
    .map(topics.set_index("topic")["top_words"])
)
selected_result = selected_documents[["name", "artists", "genre", "nmf_topic", "topic_words"]]

display(selected_result)

,name,artists,genre,nmf_topic,topic_words
7834,A New Anhedonia - Live,"[""Phosphorescent""]",Folk,6,"im, dont, know, want, feel"
8138,Roll Me Up,"[""Willie Nelson"", ""Snoop Dogg"", ""Kris Kristoff...",Country,1,"nigga, got, bitch, yeah, fuck"
7782,The Kind of Love We Make,"[""Luke Combs""]",Country,2,"love, babi, heart, true, girl"
2432,Lost Case Of Being Found,"[""Scott H. Biram""]",Country,1,"nigga, got, bitch, yeah, fuck"
1708,Galapogos - Remastered 2012,"[""The Smashing Pumpkins""]",Rock,0,"time, come, day, away, life"


## a) BERTopic - Avaliar a quantidade de tópicos
O BERTopic é capaz de calcular a quantidade de tópicos automaticamente por meio de um método de agrupamento (por exemplo, HDBSCAN). Portanto, nesse caso, deixamos o BERTopic executar em sua configuração padrão para que ele determine a quantidade de tópicos. Aleḿ disso, usamos os textos originais aplicando apenas uma filtragem leve, sem stemmização, processo que impacta negativamente no processamento do BERTopic. Apesar disso, usamos um CountVectorize configurado para ignorar stop_words em inglês para melhorar a apresentação dos tokens de maior relevância, sem impactar o processamento do modelo.

In [84]:
documents = df["lyrics"].fillna("").astype(str).tolist()

umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

# Criar e ajustar o modelo.
model = BERTopic(
    top_n_words=5,
    vectorizer_model=CountVectorizer(stop_words="english"),
    umap_model=umap_model,
)
document_topics, _ = model.fit_transform(documents)

# Mostrar os tópicos e suas 5 palavras principais.
display(model.get_topic_info()[["Topic", "Count", "Representation"]])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11746.83it/s]


,Topic,Count,Representation
0,-1,2,"[love, lalalove, gotta, yeah, baby]"
1,0,19528,"[im, like, dont, know, love]"
2,1,199,"[christmas, santa, claus, merry, bells]"
3,2,171,"[yeah, oh, baby, love, im]"
4,3,56,"[follow, hiphop, eyes, yeah, life]"
5,4,44,"[knock, wowwar, wow, hey, crazy]"


BERTopic foi executado em suas configurações padrão. O uso do UMAP se deu apenas para garantir a reprodutibilidade do resultado, sendo os valores utilizados iguais aos descritos na documentação [Referência](https://maartengr.github.io/BERTopic/getting_started/dim_reduction/dim_reduction.html#umap).

É possível perceber que o BERTopic foi capaz de identificar 5 tópicos distintos na base. O tópico marcado como -1 agrupa os textos considerados outliers, nesse caso apenas 2.

Além disso, percebe-se que o método agrupo 19528 das 20000 músicas em um único grupo temático, mais especificamente [im, like, dont, know, love]. Esse resultado apresentou baixa diferenciação temática da base. Entretanto, a temática apresentada identificada pelo BERT no tópico 0 faz algum sentido se considerarmos que boa parte das músicas populares e presentes em aplicativos de Streaming tratam da temática de amor/relacionamento, tema possível de inferir a partir das palavras de impacto do tópico identificado. Além disso, o BERTopic foi capaz de identificar com grupo temático muito específico, o de natal.

Portanto, a falta de diversidade pode estar relacionada à própria base de dados. Outro aspecto que corrobora para essa observação é que os modelos de embbeding usados pelo BERTopic possuem limite de entrada, ou seja, é possível que partes de letras muito grandes tenham sido perdidas.

## b) BERTopic - 5 palavras mais relevantes
Na letra a) é possível verificar as 5 palalvras mais relevantes para cada tópico identificado pelo BERTopic

c) BERTopic - Identificar tópicos de 5 textos usando o modelo

In [88]:
topics_by_document = pd.Series(
    model.topics_,
    index=df.index
)

selected_indices = df.sample(5, random_state=21).index
selected = df.loc[selected_indices].copy()

selected["topic"] = topics_by_document.loc[selected_indices]

# Representa cada tópico pelas suas 5 palavras principais.
selected["topic_words"] = selected["topic"].apply(
    lambda topic_id: (
        ", ".join(word for word, _ in model.get_topic(topic_id)[:5])
        if topic_id != -1
        else "Sem tópico definido (ruído)"
    )
)

display(selected[["artists", "name", "topic", "topic_words"]])

,artists,name,topic,topic_words
7834,"[""Phosphorescent""]",A New Anhedonia - Live,0,"im, like, dont, know, love"
8138,"[""Willie Nelson"", ""Snoop Dogg"", ""Kris Kristoff...",Roll Me Up,0,"im, like, dont, know, love"
7782,"[""Luke Combs""]",The Kind of Love We Make,0,"im, like, dont, know, love"
2432,"[""Scott H. Biram""]",Lost Case Of Being Found,0,"im, like, dont, know, love"
1708,"[""The Smashing Pumpkins""]",Galapogos - Remastered 2012,0,"im, like, dont, know, love"


Como esperado, dado que o BERTopic agrupou mais de 90% da base no tópico 0, seria altamente provável que todas as músicas selecionadas aleatoriamente estariam dentro deste tópico.

## d) Comparação dos dois modelos

A partir dos resultados obtidos anteriormente, o NMF com sete tópicos apresentou melhores resultados neste experimento, principalmente pela maior diferenciação entre os documentos e pelo menor tempo de execução necessário.

O BERTopic concentrou 19.528 dos 20.000 documentos em um único tópico, representado por “im, like, dont, know, love”, um conjunto de palavras relativamente genéricas. As cinco músicas selecionadas também foram atribuídas a esse grupo, embora apresentem temáticas que variam, não sendo apenas músicas voltadas para romance, como somos levados a interpretar com base nas palalvras "like" e "love". Assim, na configuração padrão utilizada, o modelo ofereceu pouca informação para distinguir os assuntos da maior parte da base. Entretanto, identificou um grupo temático muito específico específico e interpretável relacionado ao Natal.

No caso do NMF, houve uma distribuição mais diversa das cinco músicas, tendo sido apontados quatro tópicos diferentes. A atribuição de “The Kind of Love We Make” ao tópico “love, babi, heart, true, girl” foi claramente coerente com conteúdo romântico da letra. Contudo, houveram atribuições divergentes: o tópico dominado por gírias e palavrões, atribuído a “Roll Me Up” e “Lost Case Of Being Found”, não descreve claramente seus temas, apesar dessa música ter a participação do rapper "Snoop Dogg", mas o modelo não levou isso em consideração. Portanto, a maior diversidade não equivale automaticamente, a uma maior precisão na atribuição dos temas.


Dessa forma, considerando a diversidade no agrupamento de textos aos tópicos, além do significado dos tópicos obtidos e do tempo de processamento observado (qualitativamente), foi considerado que o NMF seria mais adequado para a nossa base. Entretanto, é necessário ressaltar que o método BERTopic foi empregado sem configurações adicionasi e ajuste fino de parâmetros, o que poderia possivelmente melhor seus resultados.